In [1]:
import pandas as pd
import os
import subprocess
import textwrap

In [2]:
data = pd.read_csv("../outputs/agent_actions.csv")
data.head()

,customer_unique_id,persona,agent_action
0,0000366f3b9a7992bf8c76cfdf3221e2,Silent Loyal Customers,No Action
1,0000f6ccb0745a6a4b88665a16c9f078,High-Value Enthusiasts,No Action
2,00053a61a98854899e70ed204dd4bafe,High-Value Enthusiasts,No Action
3,0005ef4cd20d2893f0d9fbd94d3c0d97,Silent Loyal Customers,No Action
4,0006fdc98a402fceb4eb0ee528f6a8d4,Silent Loyal Customers,No Action


In [3]:
SIMULATION_LIMIT = 20  # max 10–30 is ideal

sendable_actions = [
    "Personalized Discount Offer",
    "Win-Back Campaign",
    "Loyalty Points + Feedback Request"
]

simulation_df = (
    data[data["agent_action"].isin(sendable_actions)]
    .sample(n=min(SIMULATION_LIMIT, len(data)), random_state=42)
)

simulation_df

,customer_unique_id,persona,agent_action
26533,ac229c12dbb2086d76e9365e2fb9d6ee,Emerging Loyalists,Personalized Discount Offer
11010,474ca13daf52d941cc06e5a601d86f66,Emerging Loyalists,Personalized Discount Offer
13216,55ab20e989d0c5b1525c66e47af087a9,Emerging Loyalists,Personalized Discount Offer
21672,8ca72295d8f3310a1a306a439d044cc0,Fading Customers,Win-Back Campaign
19505,7dfa4e199f407850bcc352e208564a29,Emerging Loyalists,Personalized Discount Offer
22481,922106d9e4ab35ddf6a761c6137ab813,Fading Customers,Win-Back Campaign
25883,a7c9c27c939f750a01aabf3a014051d8,Emerging Loyalists,Personalized Discount Offer
8800,392c8818b71b14227a7c701e7d54a3a1,Emerging Loyalists,Personalized Discount Offer
33559,d9d6f7cb0980240a9ca2bd9434a16616,Emerging Loyalists,Personalized Discount Offer
14240,5c32f5f0716ba398ab390a9b024ef0b9,Emerging Loyalists,Personalized Discount Offer


In [4]:
def build_prompt(persona, action):
    return f"""
You are an AI customer engagement assistant.

Customer Persona:
- {persona}

Agent Decision:
- {action}

Task:
Write a polite, professional, and empathetic customer message.
The message should:
- Match the persona tone
- Clearly reflect the agent's decision
- Be concise (4–6 lines)
- Sound like a real customer communication

Do NOT mention AI or automation.
"""

In [5]:
def generate_message(prompt):
    result = subprocess.run(
        ["ollama", "run", "llama3"],
        input=prompt,
        text=True,
        capture_output=True
    )
    return result.stdout.strip()

In [7]:
def generate_message(prompt):
    result = subprocess.run(
        [
            "ollama",
            "generate",
            "llama3",
            "--prompt",
            prompt
        ],
        capture_output=True,
        text=True
    )
    return result.stdout.strip()


In [8]:
test_prompt = """
You are a customer engagement assistant.
Write a short apology message for a delayed delivery.
"""

print(generate_message(test_prompt))

In [13]:
import subprocess

def generate_message(prompt, model="llama3"):
    process = subprocess.Popen(
        ["ollama", "run", model],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="ignore"   # <- CRITICAL FIX
    )

    stdout, stderr = process.communicate(prompt + "\n")
    return stdout.strip()

In [14]:
print(generate_message("Write a friendly apology and discount message to a loyal customer."))

Here's a sample message:

"Dear [Customer Name],

I wanted to take a moment to apologize for the recent issue you experienced with your order. We're truly sorry that our service didn't meet your expectations, and we appreciate you bringing this to our attention.

As a valued and loyal customer, we want to make things right. To show our appreciation for your patience and understanding, we'd like to offer you an exclusive 20% discount on your next purchase from us.

Simply use the code LOYAL20 at checkout to receive your discount. This is our way of saying thank you for your loyalty and for giving us another chance to get it right.

If you have any further questions or concerns, please don't hesitate to reach out. We're always here to help and value your business.

Thank you again for your understanding and for being an important part of our community. We look forward to serving you better in the future!

Best regards,
[Your Name]"

This message aims to:

* Acknowledge the issue and apol

In [15]:
simulation_df = simulation_df.sample(20, random_state=42)

In [16]:
messages = []

for _, row in simulation_df.iterrows():
    prompt = build_prompt(
        persona=row["persona"],
        action=row["agent_action"]
    )
    
    message = generate_message(prompt)

    messages.append({
        "customer_unique_id": row["customer_unique_id"],
        "persona": row["persona"],
        "agent_action": row["agent_action"],
        "message": message
    })

messages_df = pd.DataFrame(messages)
messages_df.head()

,customer_unique_id,persona,agent_action,message
0,ac229c12dbb2086d76e9365e2fb9d6ee,Emerging Loyalists,Personalized Discount Offer,"Dear [Customer],\n\nThank you for reaching out..."
1,7011c18021a7ba540f53349a324677ed,Emerging Loyalists,Personalized Discount Offer,Hi there!\n\nI hope you're having an amazing d...
2,e0aa6fa7c2438e89c95d0b9cf2f5fc71,Emerging Loyalists,Personalized Discount Offer,"Here is a polite, professional, and empathetic..."
3,474ca13daf52d941cc06e5a601d86f66,Emerging Loyalists,Personalized Discount Offer,"Here is a polite, professional, and empathetic..."
4,d9d6f7cb0980240a9ca2bd9434a16616,Emerging Loyalists,Personalized Discount Offer,Here is a message that matches the persona ton...


In [19]:
import random

INCENTIVES = {
    "Personalized Discount Offer": [
        "10% off your next order",
        "₹250 cashback on your next purchase",
        "Free shipping on your next order"
    ],
    "Win-Back Campaign": [
        "15% off comeback offer",
        "₹500 comeback voucher",
        "Free express delivery on return order"
    ],
    "Loyalty Points + Feedback Request": [
        "Double loyalty points on next order",
        "Bonus reward points for feedback",
        "VIP reward points boost"
    ],
    "No Action": [
        "A simple appreciation message"
    ]
}

def choose_incentive(action):
    return random.choice(INCENTIVES.get(action, ["Thank you"]))

In [20]:
def build_prompt(persona, action, incentive):
    return f"""
You are an AI customer engagement agent.

Customer Persona: {persona}
Selected Agent Action: {action}
Assigned Incentive (do NOT change): {incentive}

Task:
Write a short, friendly, human-sounding message (3–4 sentences max).

Rules:
- Do NOT invent a new offer
- Do NOT mention percentages unless explicitly stated
- Tone must match the persona
- Sound natural, not promotional or robotic
"""

In [21]:
messages = []

for _, row in simulation_df.iterrows():
    incentive = choose_incentive(row["agent_action"])
    
    prompt = build_prompt(
        persona=row["persona"],
        action=row["agent_action"],
        incentive=incentive
    )
    
    message = generate_message(prompt)

    messages.append({
        "customer_unique_id": row["customer_unique_id"],
        "persona": row["persona"],
        "agent_action": row["agent_action"],
        "incentive": incentive,
        "message": message
    })

messages_df = pd.DataFrame(messages)
messages_df.head()

,customer_unique_id,persona,agent_action,incentive,message
0,ac229c12dbb2086d76e9365e2fb9d6ee,Emerging Loyalists,Personalized Discount Offer,Free shipping on your next order,Hi there! I wanted to reach out and show my ap...
1,7011c18021a7ba540f53349a324677ed,Emerging Loyalists,Personalized Discount Offer,Free shipping on your next order,Hi there! We've noticed you're starting to mak...
2,e0aa6fa7c2438e89c95d0b9cf2f5fc71,Emerging Loyalists,Personalized Discount Offer,₹250 cashback on your next purchase,"""Hey there! I'm glad you're thinking about sho..."
3,474ca13daf52d941cc06e5a601d86f66,Emerging Loyalists,Personalized Discount Offer,₹250 cashback on your next purchase,"""Hey there! We're thrilled to see you're getti..."
4,d9d6f7cb0980240a9ca2bd9434a16616,Emerging Loyalists,Personalized Discount Offer,₹250 cashback on your next purchase,"Hi there! As a valued customer, we wanted to s..."


In [22]:
import os

os.makedirs("../outputs/llm_messages", exist_ok=True)

for _, row in messages_df.iterrows():
    filename = f"../outputs/llm_messages/{row['customer_unique_id']}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(row["message"])